# 02 — Training runs

5 architectures × 3 split conditions × 3 seeds = 45 runs.
Each run saves val/test probabilities to `preds.csv`; all metrics are computed later from those files.

**Setup:** dataset `paultimothymooney/chest-xray-pneumonia`, Internet **on**, accelerator **GPU T4 ×1** (or P100).

**How to run**
1. Interactive session: run everything with `SMOKE = True`. Takes ~5 min. Send the printed epoch times.
2. Set `SMOKE = False`, then **Save Version → Save & Run All (Commit)**. Runs in the background; outputs are kept.
3. If the commit stops early (12 h limit), add its output as an input to the next commit: finished runs are copied in and skipped.

In [ ]:
!pip install -q --no-deps --force-reinstall --no-cache-dir git+https://github.com/meap11/ppneu.git
try:
    import timm
except ImportError:
    !pip install -q timm

In [ ]:
SMOKE = True

SPLITS = ["official", "image_random", "grouped"]
ARCHS  = ["resnet50", "densenet121", "inception_v3", "mobilenetv3", "efficientnet_b0"]
SEEDS  = [0, 1, 2]

In [ ]:
import os, shutil, json
from pathlib import Path
import pandas as pd, torch, timm
from ppneu import audit as A, data as D, train as T

print("torch", torch.__version__, "| timm", timm.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Turn on a GPU accelerator"

root, _ = A.find_root("/kaggle/input")
manifests = {s: D.load_manifest(s) for s in SPLITS}
# the official manifest lists all 5,856 images, so it builds the full cache
cache = D.build_cache(manifests["official"], root, "/kaggle/working/cache/cache256.npy",
                      workers=os.cpu_count())
print("cache", cache.shape)

In [ ]:
# Resume: copy finished runs from any attached previous output of this notebook
OUT = "/kaggle/working/smoke" if SMOKE else "/kaggle/working/runs"
for prev in Path("/kaggle/input").glob("*/runs") if not SMOKE else []:
    for preds in prev.glob("*/*/seed*/preds.csv"):
        dst = Path(OUT) / preds.parent.relative_to(prev)
        if not (dst / "preds.csv").exists():
            shutil.copytree(preds.parent, dst, dirs_exist_ok=True)
            print("resumed", dst)

In [ ]:
if SMOKE:
    plan = [T.RunConfig(split="grouped", arch=a, seed=0, epochs=2, limit=256,
                        workers=2, out_root=OUT) for a in ARCHS]
else:
    # seeds outermost, so a partial commit still covers every model and split
    plan = [T.RunConfig(split=s, arch=a, seed=k, workers=4, out_root=OUT)
            for k in SEEDS for s in SPLITS for a in ARCHS]
print(len(plan), "runs")

for i, cfg in enumerate(plan, 1):
    print(f"\n=== {i}/{len(plan)}: {cfg.split} / {cfg.arch} / seed {cfg.seed} ===")
    meta = T.train_one(cfg, manifests[cfg.split], cache, force=SMOKE)
    print(f"best epoch {meta['best_epoch']} | val AUROC {meta['val_auc']:.4f} | "
          f"{meta['minutes']:.1f} min")

In [ ]:
summary = T.collect(OUT)
summary.to_csv(Path(OUT) / "summary.csv", index=False)
display(summary)
if not SMOKE and len(summary):
    display(summary.pivot_table(index="arch", columns="split", values="test_auc",
                                aggfunc=["mean", "std"]).round(4))

In [ ]:
# Free space: the ~0.4 GB image cache is not needed in the saved output
shutil.rmtree("/kaggle/working/cache", ignore_errors=True)
!du -sh /kaggle/working/*